# NumPy — Ejercicios Avanzados (Sesión Tutorial)
**Carrera:** Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Programación II  


> Objetivo: practicar *broadcasting* avanzado, álgebra lineal, `einsum/tensordot`, indexación booleana, simulación y rendimiento, con foco en soluciones vectorizadas (sin bucles explícitos).


In [ ]:
# Configuración
import numpy as np
import math, time

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(2025)
print("NumPy:", np.__version__)

## 1) Broadcasting avanzado
Trabaja únicamente con operaciones vectorizadas y `keepdims=True` donde corresponda.


In [ ]:
# 1.1 Estandarización por columnas (z-score) con broadcasting
X = rng.normal(loc=10, scale=3, size=(200, 5))

# TODO: calcula Z = (X - media_col) / std_col a lo largo del eje 0
media_col = ...  # TODO: shape (1, 5)
std_col   = ...  # TODO: shape (1, 5)
Z = ...          # TODO
# Verifica media≈0 y std≈1 por columna
assert np.allclose(Z.mean(axis=0), 0, atol=1e-12)
assert np.allclose(Z.std(axis=0, ddof=0), 1, atol=1e-12)
Z.shape

In [ ]:
# 1.2 Distancias euclidianas por pares (matriz NxN) sin bucles
Y = rng.normal(size=(150, 3))  # 150 puntos en R^3
# TODO: matriz D donde D[i,j] = ||Y[i]-Y[j]||_2
# Pista: usa ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b y luego sqrt
sq_norm = ...     # TODO: shape (150, 1)
G = ...           # TODO: gram matrix (Y @ Y.T)
D2 = ...          # TODO: distancias cuadradas
D = ...           # TODO: distancias
# checks
assert D.shape == (150,150)
assert np.all(D >= -1e-12)  # tolerancia numérica
D[:3, :3]

In [ ]:
# 1.3 Softmax estable a lo largo del último eje en un tensor 3D
T = rng.normal(size=(4, 5, 6))
# TODO: aplica softmax estable por eje -1 (colapsa la última dimensión)
# pista: restar el máximo por fila antes de exponentes
shift = ...       # TODO: shape (4,5,1)
exps = ...        # TODO
soft = ...        # TODO
# Verifica que cada vec. a lo largo del último eje suma 1
assert np.allclose(soft.sum(axis=-1), 1, atol=1e-9)
soft.shape

## 2) Álgebra lineal
Usa `np.linalg` de forma robusta y verifica con residuos/igualdades.


In [ ]:
# 2.1 Resolver sistema Ax=b y verificar residuo
A = rng.normal(size=(6,6))
b = rng.normal(size=(6,))
# TODO: x = solución de Ax=b (usa solve)
x = ...  # TODO
residuo = np.linalg.norm(A@x - b)
assert residuo < 1e-10
residuo

In [ ]:
# 2.2 Autovalores/vectores en matriz simétrica
S = rng.normal(size=(5,5)); S = (S + S.T)/2  # simetriza
# TODO: usa eigh (no eig) y verifica S*v ≈ λ*v
vals, vecs = ...  # TODO
check = np.linalg.norm(S @ vecs - vecs * vals)
assert check < 1e-10
vals[:3]

In [ ]:
# 2.3 Aproximación de rango bajo con SVD (k=2)
M = rng.normal(size=(30, 10))
# TODO: SVD y aproximación M_k con k=2
U, s, Vt = ...     # TODO
k = 2
Mk = ...           # TODO: U[:,:k] @ diag(s[:k]) @ Vt[:k]
err = np.linalg.norm(M - Mk) / np.linalg.norm(M)
err

## 3) `einsum` y `tensordot`
Formula contracciones tensoriales de forma explícita.


In [ ]:
# 3.1 Producto batched de matrices con einsum
# batch B=8, matrices 4x5 y 5x3 -> resultado 4x3 por batch
A = rng.normal(size=(8,4,5))
B = rng.normal(size=(8,5,3))
# TODO: C[b] = A[b] @ B[b] con einsum
C = ...  # TODO: einsum string
assert C.shape == (8,4,3)
C.shape

In [ ]:
# 3.2 Tensordot equivalente y verificación
C_td = ...  # TODO: tensordot con axes apropiados
assert np.allclose(C, C_td)
C_td.shape

In [ ]:
# 3.3 Outer product + reducción: suma_i x_i * y_i * z (broadcasting)
x = rng.normal(size=(50,))
y = rng.normal(size=(50,))
z = rng.normal(size=(10,))         # vector independiente
# Objetivo: un tensor T[j] = sum_i x_i * y_i * z_j  (resultado de shape (10,))
# TODO: usa einsum o broadcasting eficiente (sin bucles)
T = ...  # TODO
T.shape

## 4) Indexación booleana y agregaciones tipo *groupby*
Evita bucles; usa `np.where`, `np.add.at`, `np.bincount` o `take_along_axis`.


In [ ]:
# 4.1 Reemplazo de outliers por mediana (z-score)
X = rng.normal(size=1000)
X[::50] = 10  # inyecta outliers
# TODO: calcula z = (X - mean)/std y reemplaza |z|>3 por la mediana de X sin outliers
mu = ...      # TODO
sd = ...      # TODO
z = ...       # TODO
mask = ...    # TODO: booleano de outliers
med = ...     # TODO: mediana de X[~mask]
X_clean = ... # TODO: usa np.where
np.mean(np.abs((X_clean - X_clean.mean())/X_clean.std()) > 3)  # proporción outliers post-limpieza

In [ ]:
# 4.2 Agregación por claves enteras (tipo groupby) con np.add.at
keys = rng.integers(0, 5, size=200)   # grupos 0..4
vals = rng.normal(size=200)
# TODO: suma por grupo (vectorizado)
sums = np.zeros(5)
# pista: np.add.at(sums, keys, vals)
...  # TODO
# TODO: cuenta por grupo y promedio por grupo
counts = np.bincount(keys, minlength=5)
means = ...  # TODO
sums, counts, means

## 5) Convolución 1D y correlación cruzada
Usa `np.convolve` y compara con implementación vía `einsum` para ventanas.


In [ ]:
# 5.1 Promedio móvil y comparación con vectorización
s = rng.normal(loc=0, scale=1, size=100)
w = np.ones(7)/7  # ventana de 7
# TODO: promedio móvil con np.convolve (modo 'valid')
mov = ...   # TODO
# TODO: implementación alternativa con strides/einsum (solo fórmula; puedes usar einsum)
# Sugerencia de shape final: 94 (100-7+1)
# Hints: crea una matriz de ventanas con broadcasting y aplica einsum 'ij, j -> i'
# (opcional; si no lo haces, mantén sólo mov)
mov[:5]

## 6) Rendimiento: `np.where` vs `np.vectorize`
Evita `np.vectorize` cuando sea posible. Compara tiempos (aproximados).


In [ ]:
# 6.1 Comparación simple de tiempos
arr = rng.normal(size=5_000_000)

def f_scalar(x):
    # función pieza a trozos: |x| si |x|<1; x^2 en caso contrario
    return x if abs(x) < 1 else x*x

t0 = time.time()
out_where = np.where(np.abs(arr) < 1, np.abs(arr), arr*arr)
t_where = time.time() - t0

vec_f = np.vectorize(f_scalar)
t0 = time.time()
out_vec = vec_f(arr)
t_vec = time.time() - t0

print(f"np.where: {t_where:.4f}s, np.vectorize: {t_vec:.4f}s")
assert np.allclose(out_where, out_vec)

## 7) Simulación Monte Carlo: estimar π
Genera N pares (x,y) ~ U[0,1] y cuenta cuántos caen en el cuarto de círculo.


In [ ]:
# 7.1 Estimador de π (vectorizado)
N = 2_000_000
xy = rng.random((N,2))
inside = ...  # TODO: condición x^2 + y^2 <= 1
pi_hat = ...  # TODO
pi_hat

## 8) Bonus: una iteración de K-means (vectorizada)
Dados datos `X` (N×d) y centros `C` (k×d), asigna cada punto a su centro más cercano y recalcula centros.


In [ ]:
# 8.1 Una iteración de K-means
X = rng.normal(size=(500, 2))
C = rng.normal(size=(4, 2))  # k=4

# TODO: asigna etiquetas usando distancias cuadradas con broadcasting
# dist2[n,k] = ||X[n]-C[k]||^2
dist2 = ...      # TODO: shape (500,4)
labels = ...     # TODO: argmin por fila -> shape (500,)

# TODO: recalcula nuevos centros como promedio por cluster (usa bincount/add.at)
C_new = np.zeros_like(C)
counts = np.bincount(labels, minlength=C.shape[0])  # (4,)
# Suma por cluster
for d in range(X.shape[1]):
    np.add.at(C_new[:,d], labels, X[:,d])
C_new = C_new / counts[:, None]
C_new, counts

<details>
<summary><strong>Pistas rápidas</strong></summary>

- **1.1** Usa `mean(axis=0, keepdims=True)` y `std(axis=0, keepdims=True)` (evita `ddof=1`).
- **1.2** `sq_norm = (Y**2).sum(axis=1, keepdims=True)` y `G = Y @ Y.T`.
- **1.3** `shift = T.max(axis=-1, keepdims=True)`; `soft = exps / exps.sum(axis=-1, keepdims=True)`.
- **2.1** `np.linalg.solve(A,b)`.
- **2.2** `np.linalg.eigh(S)`.
- **2.3** `U,s,Vt = np.linalg.svd(M, full_matrices=False)` y luego `U[:,:k] @ (s[:k,None]*Vt[:k])`.
- **3.1** `einsum('bij,bjk->bik', A, B)`.
- **3.2** `np.tensordot(A, B, axes=([2],[1]))`.
- **3.3** `T = z * (x @ y)` o `einsum('i,i,j->j', x, y, z)`.
- **4.1** `np.where(mask, med, X)` tras calcular `mask = np.abs(z)>3`.
- **4.2** `np.add.at(sums, keys, vals)`; `means = sums / counts` con cuidado de ceros.
- **5.1** `np.convolve(s, w, mode='valid')`.
- **6.1** Evita medir con `%timeit` aquí; usa `time.time()` como en el ejemplo.
- **7.1** `inside = (xy**2).sum(axis=1) <= 1`; `pi_hat = 4 * inside.mean()`.
- **8.1** `dist2 = ((X[:,None,:]-C[None,:,:])**2).sum(axis=2)` y luego `labels = dist2.argmin(axis=1)`.
</details>
